In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# importing raw dataset
df = pd.read_csv("data.csv")
print(df.shape)

(111902, 24)


### Feature engineering and data preparation without time sensitive train test split:

In this step, the raw dataset is cleaned and transformed to create meaningful predictors for the credit risk model.

**1. Handling missing and inconsistent values**
- Created `is_new_customer` indicator where `days_since_first_loan == -1`.
- Clipped `days_since_first_loan` to remove negative values.
- Added `existing_klarna_debt_missing` flag to capture missing debt information.
- Imputed missing values in `existing_klarna_debt` with 0 (assuming missing implies no existing debt).
- Clipped negative debt values to 0, as negative debt is not economically meaningful.

**2. Date feature extraction**
From `loan_issue_date`, several time-based variables were created to capture potential seasonality in borrowing and repayment behavior:
- `loan_dayofweek`
- `loan_month`
- `loan_week`

**3. Card expiry risk**
Card expiry month and year were transformed into a more informative feature:
- `months_to_expiry` calculated relative to the loan issue date.
- `card_expiry_risk` indicator created for cards expiring within 3 months.
Raw expiry variables were dropped after feature creation.

**4. Behavioral and financial risk features**
Several ratio and trend-based features were engineered to capture customer financial behavior and repayment patterns:

- `debt_to_loan_ratio` – existing debt relative to the requested loan.
- `exposure_growth` – increase in exposure between 7 and 14 days.
- `failed_payment_ratio_3m`, `failed_payment_ratio_6m` – failed vs confirmed payments.
- `payment_reliability` – difference between confirmed and failed payments.
- `repayment_speed` – share of loan repaid within 1 month.
- `repayment_ratio_6m` – repayment share within 6 months.
- `loan_intensity` – number of active loans relative to customer history length.

**5. Dataset cleanup and target creation**
- Dropped non-informative or redundant columns (`loan_id`, `loan_issue_date`, raw card expiry fields).
- Removed duplicated rows.
- Defined the target variable as:
  
  `default = 1 if amount_outstanding_21d > 0 else 0`

- Removed leakage-related variables from predictors (`amount_outstanding_21d`, `amount_outstanding_14d`).

**6. Train–test split**
The dataset was split into training and testing sets using stratified sampling to preserve the class distribution of the target variable.



In [46]:
# handling missing variables, negative, and creating new variables

# new customers feature

df["is_new_customer"] = (df["days_since_first_loan"] == -1).astype(int)

df["days_since_first_loan"] = df["days_since_first_loan"].clip(lower=0)

# indicator for missing values in existing_klarna_debt

df["existing_klarna_debt_missing"] = df["existing_klarna_debt"].isna().astype(int)

# impute missing values in existing_klarna_debt with zero (assuming missing means no existing debt)

df["existing_klarna_debt"] = df["existing_klarna_debt"].fillna(0)

# fixing negative (~0.13% negative values) debt values (assuming negative means zero debt) / check later for positive values !!!
# negative debts cant be considered riskie.

df["existing_klarna_debt"] = df["existing_klarna_debt"].clip(lower=0)

# handle date variable
# date features / seasonality patterns in shopping behavior and loan performance

df["loan_issue_date"] = pd.to_datetime(df["loan_issue_date"])
df["loan_dayofweek"] = df["loan_issue_date"].dt.dayofweek + 1
df["loan_month"] = df["loan_issue_date"].dt.month
df["loan_week"] = df["loan_issue_date"].dt.isocalendar().week

# card expiry risk feature

df["months_to_expiry"] = (
    (df["card_expiry_year"] - df["loan_issue_date"].dt.year) * 12 +
    (df["card_expiry_month"] - df["loan_issue_date"].dt.month)
)

df["card_expiry_risk"] = (df["months_to_expiry"] <= 3).astype(int)

# dropping card expiry year and month use card_expiry_risk instead
df = df.drop(columns=["card_expiry_year","card_expiry_month","months_to_expiry"])


# creating debt to loan ratio feature (potentially a strong predictor of default risk)
# measures the proportion of existing debt relative to the new loan amount, which can indicate financial stress

df["debt_to_loan_ratio"] = df["existing_klarna_debt"] / (df["loan_amount"] + 1)

# creating exposure growth feature. Potentially a strong predictor of default risk,
# as a significant increase in exposure over a short period may indicate financial distress or aggressive borrowing behavior.
df["exposure_growth"] = df["new_exposure_14d"] - df["new_exposure_7d"]

# variables measuring payment behaviour

df["failed_payment_ratio_3m"] = (
    df["num_failed_payments_3m"] /
    (df["num_confirmed_payments_3m"] + 1)
)

df["failed_payment_ratio_6m"] = (
    df["num_failed_payments_6m"] /
    (df["num_confirmed_payments_6m"] + 1)
)

df["payment_reliability"] = (
    df["num_confirmed_payments_6m"] -
    df["num_failed_payments_6m"]
)

df["repayment_speed"] = (
    df["amount_repaid_1m"] /
    (df["loan_amount"] + 1)
)

df["repayment_ratio_6m"] = (
    df["amount_repaid_6m"] /
    (df["loan_amount"] + 1)
)

df["loan_intensity"] = df["num_active_loans"] / ( df["days_since_first_loan"] + 1)


# drop original date, load_id, and card expiry columns

df = df.drop(columns=["loan_issue_date", "loan_id"])

# droping duplicated rows

df = df.drop_duplicates(keep=False)

# creating target and predictor variables

y = (df["amount_outstanding_21d"] > 0).astype(int)
X = df.drop(columns=["amount_outstanding_21d", "amount_outstanding_14d"])

# train test split with stratification to maintain class balance between train in test sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# class distribution after train test split

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

amount_outstanding_21d
0    0.945507
1    0.054493
Name: proportion, dtype: float64
amount_outstanding_21d
0    0.945498
1    0.054502
Name: proportion, dtype: float64


#### Merchant Risk Encoding

Instead of creating multiple dummy variables for `merchant_category` and `merchant_group`, these features were converted into risk-based numerical variables using target encoding.

**1. Merchant category risk**
- Calculated the historical default rate for each `merchant_category` using the training dataset.
- Mapped this value to both train and test sets as `merchant_default_rate`.
- Unseen categories were filled with the overall default rate from the training data.
- The original `merchant_category` column was then removed.

**2. Merchant group risk (with smoothing)**
For `merchant_group`, smoothed target encoding was applied to reduce overfitting for groups with few observations.

This approach shrinks estimates for small groups toward the global default rate, improving stability.

The resulting feature `merchant_group_default_rate` replaces the original `merchant_group` column.

In [47]:
# instead of creating dummy variables representing merchants categories and groups, 
# i'll create single feature representing merchant risk

merchant_risk = (
    pd.concat([X_train["merchant_category"], y_train], axis=1)
    .groupby("merchant_category")[y_train.name]
    .mean()
)

X_train["merchant_default_rate"] = X_train["merchant_category"].map(merchant_risk)
X_test["merchant_default_rate"] = X_test["merchant_category"].map(merchant_risk)

mean_default_rate = y_train.mean()

X_train["merchant_default_rate"] = X_train["merchant_default_rate"].fillna(mean_default_rate)
X_test["merchant_default_rate"] = X_test["merchant_default_rate"].fillna(mean_default_rate)

# drop original merchant category column
X_train = X_train.drop(columns=["merchant_category"])
X_test = X_test.drop(columns=["merchant_category"])

# The same for merchant_group, thus using smoothing
train_data = pd.concat([X_train["merchant_group"], y_train], axis=1)

# group statistics
group_stats = (
    train_data
    .groupby("merchant_group")[y_train.name]
    .agg(["mean", "count"])
)

# smoothing strength
alpha = 50

# smoothed default rate
group_stats["smoothed_risk"] = (
    group_stats["count"] * group_stats["mean"] + alpha * mean_default_rate
) / (group_stats["count"] + alpha)

# mapping dictionary
merchant_group_risk = group_stats["smoothed_risk"]

# map to train and test
X_train["merchant_group_default_rate"] = X_train["merchant_group"].map(merchant_group_risk)
X_test["merchant_group_default_rate"] = X_test["merchant_group"].map(merchant_group_risk)

# fill unseen categories with global mean
X_train["merchant_group_default_rate"] = X_train["merchant_group_default_rate"].fillna(mean_default_rate)
X_test["merchant_group_default_rate"] = X_test["merchant_group_default_rate"].fillna(mean_default_rate)

# drop original merchant group column
X_train = X_train.drop(columns=["merchant_group"])
X_test = X_test.drop(columns=["merchant_group"])

In [48]:
# save preprocessed dataset
import json

data = {"dataset meta":"",
    "X_train": X_train.to_dict(orient="records"),
    "X_test": X_test.to_dict(orient="records"),
    "y_train": y_train.tolist(),
    "y_test": y_test.tolist()
}

with open("train_test_data_without_time_tts.json", "w") as f:
    json.dump(data, f)

### Adding a time-sensitive train–test split (in case it matters). Everything else remains the same as in the standard train–test split without time sensitivity.

In [50]:
# importing raw dataset

df = pd.read_csv("data.csv")

In [51]:
# handling missing variables, negative, and creating new variables

# new customers feature

df["is_new_customer"] = (df["days_since_first_loan"] == -1).astype(int)

df["days_since_first_loan"] = df["days_since_first_loan"].clip(lower=0)

# indicator for missing values in existing_klarna_debt

df["existing_klarna_debt_missing"] = df["existing_klarna_debt"].isna().astype(int)

# impute missing values in existing_klarna_debt with zero (assuming missing means no existing debt)

df["existing_klarna_debt"] = df["existing_klarna_debt"].fillna(0)

# fixing negative (~0.13% negative values) debt values (assuming negative means zero debt) / check later for positive values !!!
# negative debts cant be considered riskie.

df["existing_klarna_debt"] = df["existing_klarna_debt"].clip(lower=0)

# handle date variable
# date features / seasonality patterns in shopping behavior and loan performance

df["loan_issue_date"] = pd.to_datetime(df["loan_issue_date"])
df["loan_dayofweek"] = df["loan_issue_date"].dt.dayofweek + 1
df["loan_month"] = df["loan_issue_date"].dt.month
df["loan_week"] = df["loan_issue_date"].dt.isocalendar().week


# card expiry risk feature

df["months_to_expiry"] = (
    (df["card_expiry_year"] - df["loan_issue_date"].dt.year) * 12 +
    (df["card_expiry_month"] - df["loan_issue_date"].dt.month)
)

df["card_expiry_risk"] = (df["months_to_expiry"] <= 3).astype(int)

# dropping card expiry year and month use card_expiry_risk instead
df = df.drop(columns=["card_expiry_year","card_expiry_month","months_to_expiry"])


# creating debt to loan ratio feature (potentially a strong predictor of default risk)
# measures the proportion of existing debt relative to the new loan amount, which can indicate financial stress

df["debt_to_loan_ratio"] = df["existing_klarna_debt"] / (df["loan_amount"] + 1)

# creating exposure growth feature. Potentially a strong predictor of default risk,
# as a significant increase in exposure over a short period may indicate financial distress or aggressive borrowing behavior.
df["exposure_growth"] = df["new_exposure_14d"] - df["new_exposure_7d"]

# variables measuring payment behaviour

df["failed_payment_ratio_3m"] = (
    df["num_failed_payments_3m"] /
    (df["num_confirmed_payments_3m"] + 1)
)

df["failed_payment_ratio_6m"] = (
    df["num_failed_payments_6m"] /
    (df["num_confirmed_payments_6m"] + 1)
)

df["payment_reliability"] = (
    df["num_confirmed_payments_6m"] -
    df["num_failed_payments_6m"]
)

df["repayment_speed"] = (
    df["amount_repaid_1m"] /
    (df["loan_amount"] + 1)
)

df["repayment_ratio_6m"] = (
    df["amount_repaid_6m"] /
    (df["loan_amount"] + 1)
)

df["loan_intensity"] = df["num_active_loans"] / ( df["days_since_first_loan"] + 1)

# drop original date, load_id, and card expiry columns

df = df.drop(columns=["loan_id"])

# droping duplicated rows

df = df.drop_duplicates(keep=False)

# time sensitive train test split

df['target'] = (df["amount_outstanding_21d"] > 0).astype(int)

df = df.drop(columns=["amount_outstanding_21d",'amount_outstanding_14d'])

df = df.sort_values("loan_issue_date")

# define split point (80% train)
split_date = df["loan_issue_date"].quantile(0.8)

train = df[df["loan_issue_date"] <= split_date]
test = df[df["loan_issue_date"] > split_date]

print("Train size:", train.shape)
print("Test size:", test.shape)

target = "target"

X_train = train.drop(columns=[target,"loan_issue_date"])
y_train = train[target]

X_test = test.drop(columns=[target,"loan_issue_date"])
y_test = test[target]

# class distribution after train test split

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


Train size: (89647, 34)
Test size: (22089, 34)
target
0    0.943054
1    0.056946
Name: proportion, dtype: float64
target
0    0.955453
1    0.044547
Name: proportion, dtype: float64


In [52]:
# merchant category risk variable

merchant_risk = (
    pd.concat([X_train["merchant_category"], y_train], axis=1)
    .groupby("merchant_category")[y_train.name]
    .mean()
)

X_train["merchant_default_rate"] = X_train["merchant_category"].map(merchant_risk)
X_test["merchant_default_rate"] = X_test["merchant_category"].map(merchant_risk)

mean_default_rate = y_train.mean()

X_train["merchant_default_rate"] = X_train["merchant_default_rate"].fillna(mean_default_rate)
X_test["merchant_default_rate"] = X_test["merchant_default_rate"].fillna(mean_default_rate)

# drop original merchant category column
X_train = X_train.drop(columns=["merchant_category"])
X_test = X_test.drop(columns=["merchant_category"])


# The same for merchant_group, thus using smoothing

train_data = pd.concat([X_train["merchant_group"], y_train], axis=1)

# group statistics
group_stats = (
    train_data
    .groupby("merchant_group")[y_train.name]
    .agg(["mean", "count"])
)

# smoothing strength
alpha = 50

# smoothed default rate
group_stats["smoothed_risk"] = (
    group_stats["count"] * group_stats["mean"] + alpha * mean_default_rate
) / (group_stats["count"] + alpha)

# mapping dictionary
merchant_group_risk = group_stats["smoothed_risk"]

# map to train and test
X_train["merchant_group_default_rate"] = X_train["merchant_group"].map(merchant_group_risk)
X_test["merchant_group_default_rate"] = X_test["merchant_group"].map(merchant_group_risk)

# fill unseen categories with global mean
X_train["merchant_group_default_rate"] = X_train["merchant_group_default_rate"].fillna(mean_default_rate)
X_test["merchant_group_default_rate"] = X_test["merchant_group_default_rate"].fillna(mean_default_rate)

# drop original merchant group column
X_train = X_train.drop(columns=["merchant_group"])
X_test = X_test.drop(columns=["merchant_group"])

In [53]:
# save 
import json

data = {"dataset meta":"",
    "X_train": X_train.to_dict(orient="records"),
    "X_test": X_test.to_dict(orient="records"),
    "y_train": y_train.tolist(),
    "y_test": y_test.tolist()
}

with open("train_test_data_with+++_time_tts.json", "w") as f:
    json.dump(data, f)